In [1]:
import os
import numpy as np
import pandas as pd

# Keras 3 is installed, but this notebook's legacy imports (keras.preprocessing) can break.
# Use tf_keras (TF-Keras 2.18) utilities which are available in this environment.
import tensorflow as tf
import tf_keras
from tf_keras import layers, models

# Reproducibility (score-neutral; helps stability)
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Kaggle paths (fix incorrect ../input/train/train style paths)
DATA_DIR = "/kaggle/input/aerial-cactus-identification"
TRAIN_DIR = os.path.join(DATA_DIR, "train")
TEST_DIR = os.path.join(DATA_DIR, "test")
TRAIN_CSV = os.path.join(DATA_DIR, "train.csv")
SAMPLE_SUB = os.path.join(DATA_DIR, "sample_submission.csv")

label_frame = pd.read_csv(TRAIN_CSV)
test_frame = pd.read_csv(SAMPLE_SUB)

print("Train rows:", len(label_frame), "Test rows:", len(test_frame))
print(
    "Train dir exists:",
    os.path.isdir(TRAIN_DIR),
    "Test dir exists:",
    os.path.isdir(TEST_DIR),
)



2026-01-10 23:34:07.597943: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1768088047.611790      11 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1768088047.616061      11 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2026-01-10 23:34:07.633686: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

Train rows: 14175 Test rows: 3325
Train dir exists: True Test dir exists: True


In [2]:
# Load images into numpy arrays (same core approach as original code: in-memory arrays, scaled to [0,1])
from tf_keras.utils import load_img, img_to_array


def load_images_from_ids(ids, base_dir):
    arr = np.zeros((len(ids), 32, 32, 3), dtype=np.float32)
    for i, fname in enumerate(ids):
        p = os.path.join(base_dir, fname)
        pil = load_img(p, target_size=(32, 32))  # RGB by default
        arr[i] = img_to_array(pil)
    return arr


x_train = (
    load_images_from_ids(label_frame["id"].values, TRAIN_DIR).astype("float32") / 255.0
)
y_train = label_frame["has_cactus"].values.astype("float32")

x_test = (
    load_images_from_ids(test_frame["id"].values, TEST_DIR).astype("float32") / 255.0
)

print(
    "x_train:", x_train.shape, x_train.dtype, "y_train:", y_train.shape, y_train.dtype
)
print("x_test:", x_test.shape, x_test.dtype)



x_train: (14175, 32, 32, 3) float32 y_train: (14175,) float32
x_test: (3325, 32, 32, 3) float32


In [3]:
# Replace legacy ImageDataGenerator with an equivalent tf.data augmentation pipeline.
# This preserves the original intent: vertical_flip, horizontal_flip, zoom_range=0.1
BATCH_SIZE = 32
STEPS_PER_EPOCH = 250
EPOCHS = 100

augment = tf_keras.Sequential(
    [
        layers.RandomFlip(mode="horizontal_and_vertical", seed=SEED),
        layers.RandomZoom(
            height_factor=(-0.1, 0.1), width_factor=(-0.1, 0.1), seed=SEED
        ),
    ],
    name="augment",
)

train_ds = tf.data.Dataset.from_tensor_slices((x_train, y_train))
train_ds = train_ds.shuffle(
    buffer_size=len(x_train), seed=SEED, reshuffle_each_iteration=True
)
train_ds = train_ds.batch(BATCH_SIZE, drop_remainder=True)
train_ds = train_ds.map(
    lambda x, y: (augment(x, training=True), y), num_parallel_calls=tf.data.AUTOTUNE
)
train_ds = train_ds.prefetch(tf.data.AUTOTUNE)

test_ds = (
    tf.data.Dataset.from_tensor_slices(x_test).batch(256).prefetch(tf.data.AUTOTUNE)
)



2026-01-10 23:34:18.086794: W tensorflow/core/common_runtime/gpu/gpu_bfc_allocator.cc:47] Overriding orig_value setting because the TF_FORCE_GPU_ALLOW_GROWTH environment variable is set. Original config value was 0.
I0000 00:00:1768088058.087239      11 gpu_device.cc:2022] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 46866 MB memory:  -> device: 0, name: NVIDIA RTX A6000, pci bus id: 0000:06:00.0, compute capability: 8.6


In [4]:
# Keep core model logic/architecture the same: VGG16 base + Flatten + BN + Dense(256) + BN + Dropout + Sigmoid
from tf_keras.applications import VGG16


def get_model():
    base = VGG16(include_top=False, weights="imagenet", input_shape=(32, 32, 3))
    base.trainable = True

    set_trainable = False
    for layer in base.layers:
        if layer.name == "block5_conv3":
            set_trainable = True
        layer.trainable = bool(set_trainable)

    model = models.Sequential()
    model.add(base)
    model.add(layers.Flatten())
    model.add(layers.BatchNormalization())
    model.add(layers.Dense(256, activation="relu"))
    model.add(layers.BatchNormalization())
    model.add(layers.Dropout(0.5))
    model.add(layers.Dense(1, activation="sigmoid"))

    model.compile(loss="binary_crossentropy", optimizer="adam", metrics=["acc"])
    return model


model = get_model()
model.summary()



58889256/58889256 [==============================] - 1s 0us/step
Model: "sequential"
_________________________________________________________________
 Layer (type)                Output Shape              Param #   
 vgg16 (Functional)          (None, 1, 1, 512)         14714688  
                                                                 
 flatten (Flatten)           (None, 512)               0         
                                                                 
 batch_normalization (Batch  (None, 512)               2048      
 Normalization)                                                  
                                                                 
 dense (Dense)               (None, 256)               131328    
                                                                 
 batch_normalization_1 (Bat  (None, 256)               1024      
 chNormalization)                                                
                                                         

In [5]:
# Keras 3 removed fit_generator; use fit() directly on the tf.data dataset (same training semantics).
history = model.fit(
    train_ds,
    epochs=EPOCHS,
    steps_per_epoch=STEPS_PER_EPOCH,
    verbose=2,
)

y_predictions = model.predict(test_ds, verbose=0).reshape(-1)

# Ensure submission format exactly matches required header/columns
result = pd.DataFrame({"id": test_frame["id"].values, "has_cactus": y_predictions})
result.to_csv("submission.csv", index=False)

print("Wrote submission.csv with shape:", result.shape)
print(result.head())

Epoch 1/100


I0000 00:00:1768088061.583033      64 cuda_dnn.cc:529] Loaded cuDNN version 90300
I0000 00:00:1768088062.939117      64 service.cc:148] XLA service 0x7f3c8a62b490 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
I0000 00:00:1768088062.939155      64 service.cc:156]   StreamExecutor device (0): NVIDIA RTX A6000, Compute Capability 8.6
2026-01-10 23:34:22.944768: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:268] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1768088063.005064      64 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


250/250 - 3s - loss: 0.1990 - acc: 0.9298 - 3s/epoch - 14ms/step
Epoch 2/100
250/250 - 1s - loss: 0.1242 - acc: 0.9575 - 636ms/epoch - 3ms/step


2026-01-10 23:34:24.888856: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
2026-01-10 23:34:24.888964: I tensorflow/core/framework/local_rendezvous.cc:405] Local rendezvous is aborting with status: OUT_OF_RANGE: End of sequence
	 [[{{node IteratorGetNext}}]]
	 [[IteratorGetNext/_4]]
2026-01-10 23:34:24.888981: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 4798545450305773395
2026-01-10 23:34:24.889003: I tensorflow/core/framework/local_rendezvous.cc:424] Local rendezvous recv item cancelled. Key hash: 7095213970814933098


Wrote submission.csv with shape: (3325, 2)
                                     id  has_cactus
0  09034a34de0e2015a8a28dfe18f423f6.jpg    0.958273
1  134f04305c795d6d202502c2ce3578f3.jpg    0.999609
2  41fad8d145e6c41868ce3617e30a2545.jpg    0.998496
3  35f8a11352c8d41b6231bb33d8d09f7e.jpg    0.999216
4  b77dc902b035887cbbc01920ce0e3151.jpg    0.998874
